<a href="https://colab.research.google.com/github/saivenkat-954/Multi-Agent-AI-System-Design/blob/main/Google_GenAI_APAC_Hackathon_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import auth
auth.authenticate_user()

# --- EDIT THESE ---
PROJECT_ID = "sinuous-faculty-492419-r2"
REGION = "us-central1"
SERVICE_NAME = "agent-orchestrator"
# ------------------

!gcloud config set project {PROJECT_ID}
# Enable Cloud Run and Cloud Build APIs
!gcloud services enable run.googleapis.com cloudbuild.googleapis.com

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
Operation "operations/acat.p2-578841658226-f486a79d-805b-4bb8-9cf3-6b92f65c5d78" finished successfully.


### Granting Artifact Registry Permissions

The previous deployment failed due to `PERMISSION_DENIED` for Artifact Registry. You need to enable the Artifact Registry API and grant appropriate permissions to the authenticated user/service account.

First, ensure the Artifact Registry API is enabled:

In [4]:
!gcloud services enable artifactregistry.googleapis.com

Next, grant the `Artifact Registry Writer` role to the user or service account that authenticated via `auth.authenticate_user()`. Replace `YOUR_GCP_ACCOUNT` with the email address displayed in the previous error message (e.g., `saivenkat3954@gmail.com`).

In [5]:
# Replace 'YOUR_GCP_ACCOUNT' with your actual account email, e.g., 'saivenkat3954@gmail.com'
YOUR_GCP_ACCOUNT = "-------------" # Ensure this matches the account from the error message

!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member=user:{YOUR_GCP_ACCOUNT} \
    --role=roles/artifactregistry.writer

ERROR: Policy modification failed. For a binding with condition, run "gcloud alpha iam policies lint-condition" to identify issues in condition.
ERROR: (gcloud.projects.add-iam-policy-binding) INVALID_ARGUMENT: User ------------- does not exist.


After running the above commands, re-run the deployment command in the next cell.

In [6]:
%%writefile main.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

app = FastAPI()

# Specialist Tool 1: Database/Task Agent Logic
@tool
def task_db_tool(query: str):
    """Adds or retrieves tasks from the persistent task database."""
    return f"DB_SUCCESS: Action recorded for '{query}'"

# Specialist Tool 2: Calendar Agent Logic
@tool
def calendar_tool(event_details: str):
    """Schedules or checks events on the user calendar."""
    return f"CALENDAR_SUCCESS: Event '{event_details}' scheduled."

# Primary Orchestrator Setup
llm = ChatOpenAI(model="gpt-4-turbo")
tools = [task_db_tool, calendar_tool]
agent_executor = create_react_agent(llm, tools)

class AgentRequest(BaseModel):
    prompt: str

@app.post("/execute")
async def run_agent(request: AgentRequest):
    inputs = {"messages": [("user", request.prompt)]}
    result = await agent_executor.ainvoke(inputs)
    return {"answer": result["messages"][-1].content}

@app.get("/health")
def health():
    return {"status": "ok"}

Overwriting main.py


In [7]:
%%writefile requirements.txt
fastapi
uvicorn
langchain-openai
langgraph
pydantic

Overwriting requirements.txt


In [8]:
%%writefile Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
# Cloud Run injects the $PORT environment variable
CMD uvicorn main:app --host 0.0.0.0 --port $PORT

Overwriting Dockerfile


In [9]:
import gradio as gr
import requests

# PASTE YOUR CLOUD RUN URL HERE (e.g., https://agent-xxx.a.run.app)
DEPLOYED_URL = "https://your-service-url-here.a.run.app"

def call_deployed_agent(message, history):
    try:
        response = requests.post(
            f"{DEPLOYED_URL}/execute",
            json={"prompt": message},
            timeout=30
        )
        return response.json()["answer"]
    except Exception as e:
        return f"Error connecting to Cloud Run: {str(e)}"

# Launch the UI inside Colab
demo = gr.ChatInterface(
    fn=call_deployed_agent,
    title="Multi-Agent System (Live on Cloud Run)",
    description="I am running on Google Cloud! Ask me to manage your tasks and calendar."
)
demo.launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b2384376b24403fce0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
